In [ ]:
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl");

# We set-up a simple time-independent two-well system

We start with the usual equation of the resonant flux-biased ATS Hamiltonian in the RWA
\begin{equation}
    \begin{cases}
    \dot a = -i\beta_a'a^*b\\
    \dot b = -i\beta_b'a^2/2-i\varepsilon_d'/2-\kappa_b'b
    \end{cases}
\end{equation}
Then, we perform adiabatic elimination $\dot b = 0$ and we get $$\dot a = -i\beta_a'a^*(i\beta_b'a^2/2\kappa_b'-i\varepsilon_d'/2\kappa_b')$$

We introduce the quadratures $x=(a+a^*)/\sqrt 2$ and $p=i(a^*-a)/\sqrt 2$ 
\begin{equation}
    \begin{cases}
    \dot x = - \aleph (x^2+p^2)x - \beth x\\
    \dot p = - \aleph (x^2+p^2)p + \beth p
    \end{cases},
\end{equation}
with $\aleph = \beta_a'\beta_b'/4\kappa_b'$ and $\beth = \beta_a'\varepsilon_d'/2\kappa_b'$.

Then we can define a potential $U(x,p)$ so that $\dot x = \partial_xU$ and $\dot p = \partial_pU$ :$$U(x,p) = \aleph(x^2+p^2)^2/4+\beth(x^2-p^2)/2$$



In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2
#### Caracterization of the circuit #####
ω_a = 25.338776456203686
ω_b = 2*ω_a

φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_a^2
C_b_inv = Z_b*ω_b
E_Cb = C_b_inv/2

E_J = 37.12*2pi*1.5

#Environment
κ_b=1/10.4; # 1/ns from Leon's paper

ω_d=ω_b # Drive frequency
ω_p=2*ω_a-ω_d # Pump Frequency

ε_p = 0.2
g2 = besselj(1,ε_p)*E_J*φ_a^2*φ_b # (Leon's paper)
ε_d = 10.12*g2#0.75*g2_max

β_a_p, β_b_p = 2*E_J*sin(ε_p)*φ_a^2/(ω_a), 2*E_J*sin(ε_p)*φ_b^2/(ω_b)
ε_d_p = ε_d*φ_b/ω_b
κ_b_p = κ_b / ω_b

ℵ = β_a_p*β_b_p/(4*κ_b_p)
ℶ = β_a_p*ε_d_p/(2*κ_b_p)

In [ ]:
sqrt(2*ε_d_p/β_b_p)

In [ ]:
function equation!(du,u,params,t)
    ℵ,ℶ = params

    x = u[1]
    p = u[2]
        
    du[1] = -ℵ*(x^2+p^2)*x-ℶ*x
    du[2] = -ℵ*(x^2+p^2)*p+ℶ*p
    return nothing
end

In [ ]:
t_min = 0.0
t_max = 2/κ_b_p 
tspan = (t_min, t_max)
x_0, p_0 = 0.0, -0.001
u_0 = [x_0, p_0]
prob = ODEProblem(equation!, u_0, tspan, (ℵ,ℶ))
sol = solve(prob, Tsit5())
t = sol.t
x = [u[1] for u in sol.u]
p = [u[2] for u in sol.u]
p_stable=p;

In [ ]:
x_all = -.2:0.01:.2
p_all = -.35:0.01:.35

U = [ℵ*(x^2 + p^2)^2/4 + ℶ*(x^2 - p^2)/2 for p in p_all, x in x_all]

heatmap(x_all, p_all, U,
    colorbar_title="\n Meta-potential",
    xlabel=L"$x$",
    ylabel=L"$p$",
)
scatter!(x,p, label="Trajectory")

Since the system is only 1D ($x=0$ on the minimizing path and on the dissipative path), we will focus only on $p$. Therfore the equation of motion reads
$$\dot p = -\aleph p^3+\beth p$$

# Writing the Hamiltonian and initializing the GMAM

We now add noise to the system
\begin{equation}
    \dot p = - \aleph p^3 + \beth p + \sqrt \epsilon\eta(t).
\end{equation}

We can write the associated action: $$S=\int dt [P_p\dot p - H(p,P_p)],$$
with the Hamiltonian $$H(p,P_p) = [ - \aleph p^2+\beth]pP_p + \epsilon P_p^2$$

We parametrize time to solve a geometric problem $$\min_{\lambda\geq0}\min_p\max_P\int_0^1[p'P-\lambda^{-1}H(p,P)]ds,$$
and we add a constraint on the parametrization of $p$ : $|dp/ds|=|p'|=cst$

Inputs: $M = 100 \in \mathbb N$, two stable fixed points $\phi_a = (-\sqrt{2\varepsilon_d'/\beta_b'})$ and $\phi_b = (0)$ of the noiseless dynamics where $H(\phi_{a,b},0)=\nabla_{\bm \theta}H(\phi_{a,b},0)=0$; a path $\{\hat\phi_i^0\}_{i\in I}$ with $\hat\phi_0^0 =\phi_a$ and $\hat\phi_M^0 =\phi_b$, such that $|\hat\phi_{i+1}^0-\hat\phi_{i}^0|$ is constant in $i$; the functions $f(\mathbf u, \mathbf v)= \alpha \nabla_{\bm \phi}H - \nabla_{\bm \theta}H$, $g(\mathbf u, \mathbf v)=\alpha \nabla_{\bm \phi}H + \nabla_{\bm \theta}H$ and $\lambda(\mathbf u, \mathbf v) = \biggl[|\langle \nabla_{\bm \theta}H, \partial_s \hat\phi \rangle|+\sqrt{\max(0,\langle \nabla_{\bm \theta}H, \partial_s \hat\phi \rangle^2-4H|\partial_s\hat\phi|^2)}\biggr]/(2|\partial_s\hat\phi|^2)$; $\Delta\tau = 0.1 > 0, \alpha = 1 >0$

We have $\nabla_{\bm\phi}H = \begin{pmatrix}- 3\aleph p^2P_p +\beth P_p \end{pmatrix}$ and $\nabla_{\bm\theta}H = \begin{pmatrix} -\aleph p^3 +\beth p + 2 \epsilon P_p\end{pmatrix}$

# Let's perform the gradient descent ascent

We got that for $n\geq0$, we solve for each $i=M,M-1,\dots,1,0$ for the value of $u_i^{n+1}$ through $$u_M^{n+1} = -v_M^n+2\phi_b,$$ $$ u_i^{n+1} = \biggl[\lambda_i(u^n,v^n)u_{i+1}^{n+1}/\Delta s + f(u_i^n,v_i^n) + u_i^n/\Delta \tau\biggr]/\biggl[1/\Delta\tau + \lambda_i(u^n,v^n)/\Delta s\biggr] , \qquad i=M-1,\dots,1,0$$

For the value of $v_i^{n+1}$ through $$v_0^{n+1} = -u_0^{n+1}+2\phi_a,$$ $$ v_{i}^{n+1} = \biggl[\lambda_i(u^{n+1},v^n)v_{i-1}^{n+1}/\Delta s + g(u_i^{n+1},v_i^n)+v_i^n/\Delta\tau\biggr]/\biggl[1/\Delta\tau + \lambda_i(u^{n+1},v^n)/\Delta s\biggr] , \qquad i=1,\dots,M-1,M$$

In [ ]:
M = 200                    # start smaller, then increase
ϵ = 4e-3
ϕ_a = -sqrt(2*ε_d_p/β_b_p)
ϕ_b = 0
ϕ_grid = collect(range(ϕ_a, ϕ_b, length=M+1))

α = 0.01
θ_best = @. (ℵ*ϕ_grid^3 - ℶ*ϕ_grid) / ϵ   # = -b(ϕ)/ϵ
θ_init = zeros(M+1)#θ_best#
#θ_init[1] = 0.0
#θ_init[end] = 0.0

u = ϕ_grid .+ α .* θ_init
v = ϕ_grid .- α .* θ_init

Δs = 1 / (M)
Δτ = 1e-2
N_iter = 1e3

lerp(a,b,t) = (1-t)*a + t*b

for n in 1:N_iter
    # ---------- pass 1: build λ(u^n,v^n), update u ----------
    p  = (u .+ v) ./ 2
    Pp = (u .- v) ./ (2α)

    gradθ = @. (-ℵ*p^3 + ℶ*p) + 2*ϵ*Pp
    gradϕ = @. (-3*ℵ*p^2 + ℶ) * Pp
    Hvals = @. ((-ℵ*p^3 + ℶ*p) * Pp) + ϵ*Pp^2
    fvals = @. α*gradϕ - gradθ

    dsϕ = zeros(M+1)
    dsϕ[1] = (p[2]-p[1])/Δs
    for i in 2:M
        dsϕ[i] = (p[i+1]-p[i-1])/(2Δs)
    end
    dsϕ[end] = (p[end]-p[end-1])/Δs
    norm2 = @. dsϕ^2
    dot_term = @. gradθ[1:end]*dsϕ
    disc = @. dot_term^2 - 4*(Hvals[1:end])*norm2
    λ_vals = @. (abs(dot_term) + sqrt(max(0.0, disc))) / (2*max(norm2, 1e-14))

    λmax = maximum(λ_vals)

    u_new = similar(u)
    u_new[end] = -v[end] + 2*ϕ_b
    @inbounds for i in M:-1:1
        u_new[i] = u[i] + (λ_vals[i] * (u_new[i+1]-u[i])/Δs + fvals[i+1]) * Δτ
    end
    # ---------- robust 1D reparam on fixed φ-grid ----------
    ϕ = (u_new .+ v) ./ 2
    θ = (u_new .- v) ./ (2α)

    θr = zeros(M+1)
    for j in 2:M
        φt = ϕ_grid[j]
        k = searchsortedfirst(ϕ, φt)
        if k <= 1
            θr[j] = θ[1]
        elseif k > M+1
            θr[j] = θ[end]
        else
            φ0, φ1 = ϕ[k-1], ϕ[k]
            τ = abs(φ1-φ0) < 1e-14 ? 0.0 : (φt-φ0)/(φ1-φ0)
            θr[j] = lerp(θ[k-1], θ[k], τ)
        end
    end
    θr[1] = 0
    θr[end] = 0

    u_new = ϕ_grid .+ α .* θr
    v_new = ϕ_grid .- α .* θr

    # ---------- pass 2: build λ(u^{n+1},v^n), update v ----------
    p  = (u_new .+ v) ./ 2
    Pp = (u_new .- v) ./ (2α)

    gradθ = @. (-ℵ*p^3 + ℶ*p) + 2*ϵ*Pp
    gradϕ = @. (-3*ℵ*p^2 + ℶ) * Pp
    Hvals = @. ((-ℵ*p^3 + ℶ*p) * Pp) + ϵ*Pp^2
    gvals = @. α*gradϕ + gradθ

    dsϕ = zeros(M+1)
    dsϕ[1] = (p[2]-p[1])/Δs
    for i in 2:M
        dsϕ[i] = (p[i+1]-p[i-1])/(2Δs)
    end
    dsϕ[end] = (p[end]-p[end-1])/Δs
    norm2 = @. dsϕ^2
    dot_term = @. gradθ[1:end]*dsϕ
    disc = @. dot_term^2 - 4*(Hvals[1:end])*norm2
    λ_vals = @. (abs(dot_term) + sqrt(max(0.0, disc))) / (2*max(norm2, 1e-14))

    v_new = similar(v)
    v_new[1] = -u_new[1] + 2*ϕ_a
    @inbounds for i in 2:M+1
        λi = λ_vals[i]
        v_new[i] = v[i] - (λi * (v[i]-v_new[i-1])/Δs - gvals[i]) * Δτ
    end

    # ---------- robust 1D reparam on fixed φ-grid ----------
    ϕ = (u_new .+ v_new) ./ 2
    θ = (u_new .- v_new) ./ (2α)

    θr = zeros(M+1)
    for j in 2:M
        φt = ϕ_grid[j]
        k = searchsortedfirst(ϕ, φt)
        if k <= 1
            θr[j] = θ[1]
        elseif k > M+1
            θr[j] = θ[end]
        else
            φ0, φ1 = ϕ[k-1], ϕ[k]
            τ = abs(φ1-φ0) < 1e-14 ? 0.0 : (φt-φ0)/(φ1-φ0)
            θr[j] = lerp(θ[k-1], θ[k], τ)
        end
    end
    θr[1] = 0
    θr[end] = 0

    u = ϕ_grid .+ α .* θr
    v = ϕ_grid .- α .* θr

    if n % 200 == 1
        pchk = (u .+ v) ./ 2
        Pchk = (u .- v) ./ (2α)
        Hchk = @. ((-ℵ*pchk^3 + ℶ*pchk) * Pchk) + ϵ*Pchk^2
        println("iter=$n, max|H|=", maximum(abs.(Hchk)))
    end
end

p = (u .+ v) ./ 2
P = (u .- v) ./ (2α)


In [ ]:
phi_all = -.25:0.001:.01
theta_all = -.1:0.1:7
function H(p,P_p,params)
    ℵ, ℶ = params
    return P_p*(-ℵ*(p^2)*p + ℶ*p) +
           ϵ*P_p^2
end
H_val = [H(p, P_p, (ℵ, ℶ)) for p in phi_all, P_p in theta_all]


heatmap(theta_all,phi_all, H_val)
xlabel!(L"$P_p$")
ylabel!(L"$p$")
contour!(theta_all, phi_all, H_val; levels=[0.0], c=:white, lw=2)
plot!(zeros(length(p_stable)), p_stable, c="red", ls=:dash, label="Noise-free dissipation")
plot!(θ_best, ϕ_grid, label="Optimal path", ls=:dash)
scatter!(P,p, label="Solution of the GDA", ms=1)
xlims!(minimum(theta_all), 3)
ylims!(minimum(phi_all), -0.2)

In [ ]:
scatter(p, ms=.5) 

In [ ]:
Hvalues = @. ((-ℵ*p^3 + ℶ*p) * P) + ϵ*P^2
plot(1:(M+1), Hvalues)
xlabel!("Path")
ylabel!("Hamiltonian")

# We try $H^2$

In [ ]:
M = 300                    # start smaller, then increase
ϵ = 1e-3
ϕ_a = -sqrt(2*ε_d_p/β_b_p)
ϕ_b = 0.0
ϕ_grid = collect(range(ϕ_a, ϕ_b, length=M+1))

α = 1e-2
θ_best = @. (ℵ*ϕ_grid^3 - ℶ*ϕ_grid) / ϵ   # = -b(ϕ)/ϵ
θ_init = θ_best#zeros(M+1)#
θ_init[1] = 0.0
θ_init[end] = 0.0

u = ϕ_grid .+ α .* θ_init
v = ϕ_grid .- α .* θ_init

Δs = 1 / (M)
Δτ = 1e-2
N_iter = 1e3

lerp(a,b,t) = (1-t)*a + t*b

for n in 1:N_iter
    # ---------- pass 1: build λ(u^n,v^n), update u ----------
    p  = (u .+ v) ./ 2
    Pp = (u .- v) ./ (2α)                    # IMPORTANT

    gradθ = @. 2*((-ℵ*p^3 + ℶ*p) + 2*ϵ*Pp)*((-ℵ*p^3 + ℶ*p) * Pp + ϵ*Pp^2)
    gradϕ = @. 2*(-3*ℵ*p^2 + ℶ) * Pp * ((-ℵ*p^3 + ℶ*p) * Pp + ϵ*Pp^2)
    Hvals = @. ((-ℵ*p^3 + ℶ*p) * Pp + ϵ*Pp^2)^2
    fvals = @. α*gradϕ - gradθ
    fmid = (fvals[1:end-1] + fvals[2:end]) ./ 2

    dsϕ = diff(p) ./ Δs
    norm2 = @. dsϕ^2
    dot_term = @. gradθ[1:end-1] * dsϕ
    disc = @. dot_term^2 - 4*Hvals[1:end-1]*norm2
    λ_vals = @. (abs(dot_term) + sqrt(max(0.0, disc))) / (2*max(norm2, 1e-14))

    λmax = maximum(λ_vals)

    u_new = similar(u)
    u_new[end] = -v[end] + 2*ϕ_b
    @inbounds for i in M:-1:1
        u_new[i] = u[i] + (λ_vals[i] * (u_new[i+1]-u[i])/Δs + fmid[i]) * Δτ
    end

    # ---------- pass 2: build λ(u^{n+1},v^n), update v ----------
    p  = (u_new .+ v) ./ 2
    Pp = (u_new .- v) ./ (2α)

    gradθ = @. 2*((-ℵ*p^3 + ℶ*p) + 2*ϵ*Pp)*((-ℵ*p^3 + ℶ*p) * Pp + ϵ*Pp^2)
    gradϕ = @. 2*(-3*ℵ*p^2 + ℶ) * Pp * ((-ℵ*p^3 + ℶ*p) * Pp + ϵ*Pp^2)
    Hvals = @. ((-ℵ*p^3 + ℶ*p) * Pp + ϵ*Pp^2)^2
    gvals = @. α*gradϕ + gradθ
    gmid = (gvals[1:end-1] + gvals[2:end]) ./ 2

    dsϕ = diff(p) ./ Δs
    norm2 = @. dsϕ^2
    dot_term = @. gradθ[1:end-1] * dsϕ          # consistent indexing
    disc = @. dot_term^2 - 4*Hvals[1:end-1]*norm2
    λ_vals = @. (abs(dot_term) + sqrt(max(0.0, disc))) / (2*max(norm2, 1e-14))

    v_new = similar(v)
    v_new[1] = -u_new[1] + 2*ϕ_a
    @inbounds for i in 2:M+1
        λi = λ_vals[i-1]
        v_new[i] = v[i] - (λi * (v[i]-v_new[i-1])/Δs - gmid[i-1]) * Δτ
    end

    # ---------- robust 1D reparam on fixed φ-grid ----------
    ϕ = (u_new .+ v_new) ./ 2
    θ = (u_new .- v_new) ./ (2α)

    perm = sortperm(ϕ)
    ϕs = ϕ[perm]
    θs = θ[perm]

    θr = zeros(M+1)
    for j in 2:M
        φt = ϕ_grid[j]
        k = searchsortedfirst(ϕs, φt)
        if k <= 1
            θr[j] = θs[1]
        elseif k > M+1
            θr[j] = θs[end]
        else
            φ0, φ1 = ϕs[k-1], ϕs[k]
            τ = abs(φ1-φ0) < 1e-14 ? 0.0 : (φt-φ0)/(φ1-φ0)
            θr[j] = lerp(θs[k-1], θs[k], τ)
        end
    end
    θr[1] = θ[1]
    θr[end] = θ[end]

    u = ϕ_grid .+ α .* θr
    v = ϕ_grid .- α .* θr

    if n % 200 == 1
        pchk = (u .+ v) ./ 2
        Pchk = (u .- v) ./ (2α)
        Hchk = @. (((-ℵ*pchk^3 + ℶ*pchk) * Pchk) + ϵ*Pchk^2)^2
        println("iter=$n, max|H|=", maximum(abs.(Hchk)))
    end
end

p = (u .+ v) ./ 2
P = (u .- v) ./ (2α);


In [ ]:
phi_all = -.25:0.001:.01
theta_all = -1:0.1:6
function H(p,P_p,params)
    ℵ, ℶ = params
    return sqrt((P_p*(-ℵ*(p^2)*p + ℶ*p) +
           ϵ*P_p^2)^2)
end
H_val = [H(p, P_p, (ℵ, ℶ)) for p in phi_all, P_p in theta_all]


heatmap(theta_all,phi_all, H_val)
xlabel!(L"$P_p$")
ylabel!(L"$p$")
contour!(theta_all, phi_all, H_val; levels=[0.0], c=:white, lw=2)
plot!(zeros(length(p_stable)), p_stable, c="red", ls=:dash, label="Noise-free dissipation")
plot!(P,p, label="Solution of the GDA")
plot!(θ_best, ϕ_grid, label="Optimal path", ls=:dash)
xlims!(minimum(theta_all), maximum(theta_all))

# Everything inside functions

In [ ]:
@inline lerp(a, b, t) = (1 - t) * a + t * b

function reparametrize_φ_alone(φ::AbstractVector{T}; epslen::T = T(1e-14)) where {T<:Real}
    N = length(φ)
    @assert N ≥ 2 "Need at least 2 points"

    s = zeros(T, N)
    @inbounds for i in 2:N
        di = abs(φ[i] - φ[i-1])
        s[i] = s[i-1] + (di > epslen ? di : zero(T))
    end

    L = s[end]
    sr = range(zero(T), L, length=N)

    φr = similar(φ)

    @inbounds for j in 1:N
        sj = sr[j]
        i = searchsortedfirst(s, sj)

        if i <= 1
            φr[j] = φ[1]
        elseif i > N
            φr[j] = φ[N]
        else
            s0, s1 = s[i-1], s[i]
            t = (s1 - s0) ≤ epslen ? zero(T) : (sj - s0) / (s1 - s0)
            φr[j] = lerp(φ[i-1], φ[i], t)
        end
    end

    return φr
end

function reparametrization_arc_length(
    φ::AbstractVector{T},
    payload::AbstractVector{T},
    epslen::T = T(1e-14),
) where {T<:Real}

    N = length(φ)
    @assert length(payload) == N
    @assert N ≥ 2 "Need at least 2 points"

    s = zeros(T, N)
    @inbounds for i in 2:N
        di = abs(φ[i] - φ[i-1])
        s[i] = s[i-1] + (di > epslen ? di : zero(T))
    end

    L = s[end]
    sr = range(zero(T), L, length=N)

    φr = similar(φ)
    payloadr = similar(payload)

    @inbounds for j in 1:N
        sj = sr[j]
        i = searchsortedfirst(s, sj)

        if i <= 1
            φr[j] = φ[1]
            payloadr[j] = payload[1]
        elseif i > N
            φr[j] = φ[N]
            payloadr[j] = payload[N]
        else
            s0, s1 = s[i-1], s[i]
            t = (s1 - s0) ≤ epslen ? zero(T) : (sj - s0) / (s1 - s0)

            φr[j] = lerp(φ[i-1], φ[i], t)
            payloadr[j] = lerp(payload[i-1], payload[i], t)
        end
    end

    return φr, payloadr
end

function reparametrization(u_new, v_new, α, M, ϕ_a, ϕ_b)
    N = length(u_new)
    @assert length(v_new) == N
    @assert N == M + 1

    ϕ = (u_new .+ v_new) ./ 2
    θ = (u_new .- v_new) ./ (2 * α)

    ϕr, θr = reparametrization_arc_length(ϕ, θ)

    θr[1] = 0.0
    θr[end] = 0.0
    ϕr[1] = ϕ_a
    ϕr[end] = ϕ_b

    u = ϕr .+ α .* θr
    v = ϕr .- α .* θr
    return u, v
end

function build_lambda_and_src(u, v, α, Δs, params; sign::Int)
    N = length(u)
    @assert length(v) == N

    ϕ = (u .+ v) ./ 2
    θ = (u .- v) ./ (2 * α)

    gθ = similar(u)
    gϕ = similar(u)
    H  = similar(u)

    @inbounds for i in 1:N
        gθ[i] = grad_theta_H(ϕ[i], θ[i], params)
        gϕ[i] = grad_phi_H(ϕ[i], θ[i], params)
        H[i]  = H_vals(ϕ[i], θ[i], params)
    end

    src = α .* gϕ .+ sign .* gθ

    dϕ = similar(u)
    @inbounds begin
        dϕ[1] = (ϕ[2] - ϕ[1]) / Δs
        for i in 2:N-1
            dϕ[i] = (ϕ[i+1] - ϕ[i-1]) / (2 * Δs)
        end
        dϕ[N] = (ϕ[N] - ϕ[N-1]) / Δs
    end

    λ = similar(u)
    @inbounds for i in 1:N
        norm2 = dϕ[i]^2
        dotterm = gθ[i] * dϕ[i]
        disc = dotterm^2 - 4 * H[i] * norm2
        denom = 2 * max(norm2, 1e-14)
        λ[i] = (abs(dotterm) + sqrt(max(0.0, disc))) / denom
    end

    return λ, src
end

function update_u(u, v, α, Δs, Δτ0, ϕ_b, params)
    N = length(u)
    M = N - 1

    λ, f = build_lambda_and_src(u, v, α, Δs, params; sign = -1)

    λmax = maximum(λ)
    Δτ = Δτ0

    u_new = similar(u)

    u_new[end] = -v[end] + 2 * ϕ_b
    u_new[M] = (u[M] / Δτ + (λ[M] / Δs) * u_new[M+1] + f[M]) / (1 / Δτ + λ[M] / Δs)

    @inbounds for i in M-1:-1:1
        u_new[i] = (u[i] / Δτ + (λ[i] / (2 * Δs)) * (-u_new[i+2] + 4 * u_new[i+1]) + f[i]) /
                   (1 / Δτ + 3 * λ[i] / (2 * Δs))
    end

    return u_new, Δτ
end

function update_v(u_new, v, α, Δs, Δτ, ϕ_a, params)
    N = length(v)
    M = N - 1

    λ, g = build_lambda_and_src(u_new, v, α, Δs, params; sign = +1)

    v_new = similar(v)

    v_new[1] = -u_new[1] + 2 * ϕ_a
    v_new[2] = (v[2] / Δτ + (λ[2] / Δs) * v_new[1] + g[2]) / (1 / Δτ + λ[2] / Δs)

    @inbounds for i in 3:M+1
        λi = λ[i]
        v_new[i] = (v[i] / Δτ + (λi / (2 * Δs)) * (4 * v_new[i-1] - v_new[i-2]) + g[i]) /
                   (1 / Δτ + 3 * λi / (2 * Δs))
    end

    return v_new
end

function compute_cost(u, v, α, Δs, params)
    N = length(u)
    @assert length(v) == N

    ϕ = (u .+ v) ./ 2
    θ = (u .- v) ./ (2 * α)

    λ, _ = build_lambda_and_src(u, v, α, Δs, params; sign = -1)

    H = similar(u)
    @inbounds for i in 1:N
        H[i] = H_vals(ϕ[i], θ[i], params)
    end

    cost = zero(eltype(u))

    @inbounds for i in 1:N-1
        dϕds = (ϕ[i+1] - ϕ[i]) / Δs
        θmid = (θ[i] + θ[i+1]) / 2
        dotval = dϕds * θmid

        Hλ_i   = H[i]   / max(λ[i],   1e-14)
        Hλ_ip1 = H[i+1] / max(λ[i+1], 1e-14)
        Hλ_mid = 0.5 * (Hλ_i + Hλ_ip1)

        cost += Δs * (dotval - Hλ_mid)
    end

    return cost
end

function residuals(u, v, α, M, params)
    Δs = 1 / M
    N = M + 1

    # λ and source terms on the converged state
    λu, f = build_lambda_and_src(u, v, α, Δs, params; sign = -1)
    λv, g = build_lambda_and_src(u, v, α, Δs, params; sign = +1)

    Ru = zeros(eltype(u), N)
    Rv = zeros(eltype(v), N)

    # u-equation residuals
    # boundary point u[end] is imposed exactly, so we usually skip it
    Ru[M] = λu[M] * (u[M] - u[M+1]) / Δs - f[M]

    @inbounds for i in 1:M-1
        Ru[i] = λu[i] * (3u[i] - 4u[i+1] + u[i+2]) / (2 * Δs) - f[i]
    end

    # v-equation residuals
    # boundary point v[1] is imposed exactly, so we usually skip it
    Rv[2] = λv[2] * (v[2] - v[1]) / Δs - g[2]

    @inbounds for i in 3:N
        Rv[i] = λv[i] * (3v[i] - 4v[i-1] + v[i-2]) / (2 * Δs) - g[i]
    end

    # ignore imposed boundary nodes
    res_nodewise = zeros(eltype(u), N)
    @inbounds for i in 2:M
        res_nodewise[i] = abs(Ru[i]) + abs(Rv[i])
    end

    res_max = maximum(res_nodewise[2:M])

    return res_max
end

In [ ]:
# ---------------- basic physics functions ----------------
function grad_phi_H(ϕ::Float64, θ::Float64, params)
    p = ϕ
    P_p = θ
    ϵ,ℵ,ℶ = params
    grad =@. -ℵ*(3p^2*P_p)+ℶ*P_p
    return(grad)
end

function grad_theta_H(ϕ::Float64, θ::Float64, params)
    p = ϕ
    P_p = θ
    ϵ,ℵ,ℶ = params
    grad =@. -ℵ*(p^2)p+ℶ*p+2*ϵ*P_p
    return(grad)
end

function H_vals(ϕ::Float64, θ::Float64, params)
    p = ϕ
    P_p = θ
    ϵ,ℵ,ℶ = params
    return @. P_p*(-ℵ*(p^2)*p + ℶ*p) +
           ϵ*(P_p^2)
end

In [ ]:
α = 0.01
M = 200                   # start smaller, then increase
ϵ = 1e-3*3
params = (ϵ, ℵ, ℶ)
ϕ_a = -sqrt(2*ε_d_p/β_b_p)
ϕ_b = 0.0
ϕ_grid = collect(range(ϕ_a, ϕ_b, length=M+1))
ϕ_init = ϕ_grid
θ_best = @. (ℵ*ϕ_grid^3 - ℶ*ϕ_grid) / ϵ   # = -b(ϕ)/ϵ
θ_init = zeros(M+1)#θ_best

u = ϕ_init .+ α .* θ_init
v = ϕ_init .- α .* θ_init

Δs = 1 / (M)
Δτ0 = 1e-1
N_iter = 1e2
N_compute_action = N_iter > 100 ? 100 : N_iter
idx_compute_action = div(N_iter,N_compute_action)
S_list = []
max_res_list = []
append!(max_res_list,residuals(u,v,α,M, params))
for n in 1:N_iter
    u_new, Δτ = update_u(u, v, α, Δs, Δτ0, ϕ_b, params)
    v_new     = update_v(u_new, v, α, Δs, Δτ, ϕ_a, params)
    ϕ = (u_new .+ v_new) ./ 2
    θ = (u_new .- v_new) ./ (2α)
    θ[1] = 0
    θ[end] = θ_best[end]

    u = ϕ_grid .+ α .* θ
    v = ϕ_grid .- α .* θ
    u, v      = reparametrization(u_new, v_new, α, M, ϕ_a, ϕ_b)
    if mod(n,idx_compute_action)==0
        S = compute_cost(u, v, α, Δs, params)
        append!(S_list, S)
        append!(max_res_list,residuals(u,v,α,M, params))
        ϕ_chk = (u .+ v) ./ 2
        θ_chk = (u .- v) ./ (2α)
        
        Hchk = [H_vals(ϕ_chk[i], θ_chk[i], params) for i in 1:(M+1)]
        println("iter=$n, max|H|=", maximum(abs.(Hchk)))
    end
end
ϕ = (u .+ v) ./ 2
θ = (u .- v) ./ (2α);
p = ϕ[:,1]
P = θ[:,1];


In [ ]:
plot(0:idx_compute_action:N_iter ,max_res_list, yscale=:log10, label="")
xlabel!("Optimization step")
ylabel!("Max Sum Res")

In [ ]:
phi_all = -.3:0.001:.1
theta_all = -1:0.1:10
function H(p,P_p,params)
    ℵ, ℶ = params
    return P_p*(-ℵ*(p^2)*p + ℶ*p) +
           ϵ*P_p^2
end
H_val = [H(p, P_p, (ℵ, ℶ)) for p in phi_all, P_p in theta_all]


heatmap(theta_all,phi_all, H_val)
xlabel!(L"$P_p$")
ylabel!(L"$p$")
contour!(theta_all, phi_all, H_val; levels=[0.0], c=:white, lw=2)
plot!(zeros(length(p_stable)), p_stable, c="red", ls=:dash, label="Noise-free dissipation")
plot!(θ_best, ϕ_grid, label="Optimal path", ls=:dash)
scatter!(P,p, label="Solution of the GDA", ms=1)
xlims!(minimum(theta_all), 3)

In [ ]:
H_fin_array = [H_vals(ϕ[i], θ[i], params) for i in 1:size(ϕ,1)]
plot(1:(M+1), H_fin_array)
xlabel!("Path")
ylabel!("Hamiltonian")

In [ ]:
max_abs_H_list = []
ℵ_list = []
E_J_ratio_list = 0.1:0.01:4
for E_J_ratio in E_J_ratio_list
    E_J = 37*2*pi*E_J_ratio
    #Environment
    κ_b=1/10.4; # 1/ns from Leon's paper

    ω_d=ω_b # Drive frequency
    ω_p=2*ω_a-ω_d # Pump Frequency

    ε_p = 0.2
    g2 = besselj(1,ε_p)*E_J*φ_a^2*φ_b # (Leon's paper)
    ε_d = 10.12*g2#0.75*g2_max

    β_a_p, β_b_p = 2*E_J*sin(ε_p)*φ_a^2/(ω_a), 2*E_J*sin(ε_p)*φ_b^2/(ω_b)
    ε_d_p = ε_d*φ_b/ω_b
    κ_b_p = κ_b / ω_b

    ℵ = β_a_p*β_b_p/(4*κ_b_p)
    push!(ℵ_list, ℵ)
    ℶ = β_a_p*ε_d_p/(2*κ_b_p)
    α = 0.01
    M = 200                   # start smaller, then increase
    ϵ = 1e-3
    params = (ϵ, ℵ, ℶ)
    ϕ_a = -sqrt(2*ε_d_p/β_b_p)
    ϕ_b = 0.0
    ϕ_grid = collect(range(ϕ_a, ϕ_b, length=M+1))
    ϕ_init = ϕ_grid
    θ_best = @. (ℵ*ϕ_grid^3 - ℶ*ϕ_grid) / ϵ   # = -b(ϕ)/ϵ
    θ_init = zeros(M+1)#θ_best

    u = ϕ_init .+ α .* θ_init
    v = ϕ_init .- α .* θ_init

    Δs = 1 / (M)
    Δτ0 = 1e-1
    N_iter = 1e2
    N_compute_action = N_iter > 100 ? 100 : N_iter
    idx_compute_action = div(N_iter,N_compute_action)
    S_list = []
    max_res_list = []
    append!(max_res_list,residuals(u,v,α,M, params))
    for n in 1:N_iter
        u_new, Δτ = update_u(u, v, α, Δs, Δτ0, ϕ_b, params)
        v_new     = update_v(u_new, v, α, Δs, Δτ, ϕ_a, params)
        ϕ = (u_new .+ v_new) ./ 2
        θ = (u_new .- v_new) ./ (2α)
        θ[1] = 0
        θ[end] = θ_best[end]

        u = ϕ_grid .+ α .* θ
        v = ϕ_grid .- α .* θ
        u, v      = reparametrization(u_new, v_new, α, M, ϕ_a, ϕ_b)
    end
    ϕ = (u .+ v) ./ 2
    θ = (u .- v) ./ (2α);
    H_fin_array = [H_vals(ϕ[i], θ[i], params) for i in 1:size(ϕ,1)]
    push!(max_abs_H_list, maximum(abs.(H_fin_array)))
end

In [ ]:
plot(ℵ_list./ϵ, max_abs_H_list, yscale=:log10, label="")
xlabel!(L"$\aleph/\epsilon$")
ylabel!(L"\max|H|")
vline!([975], label="")
savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/switching_rate/evolution_issue_renormalization_toy_model_20_03_2026.pdf")

In [ ]:
65/sqrt(1.5)